# 56. The Safety Stock & Reorder Point (Q,r) Policy Problem

## Tier 2: Priority-Based Heuristic Algorithm

### Key Assumptions
- Multiple optimization strategies can be prioritized based on item criticality
- Cost ratios and variability measures determine item priority scores
- Priority queue structure enables efficient strategy selection
- Different strategies (cost-focused, service-focused, balanced) suit different scenarios

### Approach (Step-by-Step)
1. **Calculate criticality score** based on cost ratios and variability measures
2. **Initialize priority queue** with different optimization strategies
3. **Apply selected strategy** based on priority ranking
4. **Evaluate total cost** for each strategy variant
5. **Select best solution** across all evaluated strategies
6. **Compare results** with mathematical formulation baseline

### What to Look For in the Results
- Criticality score that reflects item importance and risk profile
- Strategy selection that adapts to item characteristics
- Cost comparison between different optimization approaches
- Performance trade-offs between cost minimization and service level

### Concrete Example: CardioStabil Medication
- High-value pharmaceutical with critical service requirements
- Stockout costs significantly exceed holding costs
- Demand and lead time variability create complex optimization landscape
- Multiple viable strategies with different cost-service trade-offs

In [1]:
# Import required libraries for priority-based heuristic
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import math
import heapq
from dataclasses import dataclass
from typing import Dict, List, Tuple

# Set plotting style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully for priority-based heuristic")

Libraries imported successfully for priority-based heuristic


Libraries imported successfully for priority-based heuristic


Libraries imported successfully for priority-based heuristic


Libraries imported successfully for priority-based heuristic


Libraries imported successfully for priority-based heuristic


In [ ]:
# Define the Priority-Based QR Policy class
@dataclass
class InventoryParameters:
    """Data class for inventory problem parameters"""
    annual_demand: float
    lead_time_mean: float
    lead_time_std: float
    demand_std: float
    holding_cost_rate: float
    unit_cost: float
    ordering_cost: float
    stockout_cost: float
    service_level: float
    
    def __post_init__(self):
        self.daily_demand = self.annual_demand / 365
        self.holding_cost = self.holding_cost_rate * self.unit_cost
        self.z_score = stats.norm.ppf(self.service_level)

class PriorityBasedQRPolicy:
    """Priority-based algorithm for (Q,r) policy optimization"""
    
    def __init__(self, params: InventoryParameters):
        self.params = params
        self.criticality_score = self.calculate_criticality_score()
    
    def calculate_criticality_score(self) -> float:
        """
        Compute priority score based on costs and variabilities.
        Higher scores indicate more critical items requiring careful optimization.
        
        Formula: Criticality = (stockout_cost / holding_cost) × (1 + demand_cv) × (1 + lead_time_cv)
        """
        # Calculate cost ratio (stockout cost vs holding cost)
        cost_ratio = self.params.stockout_cost / self.params.holding_cost
        
        # Calculate variability coefficients
        demand_cv = self.params.demand_std / self.params.daily_demand
        lead_time_cv = self.params.lead_time_std / self.params.lead_time_mean
        
        # Compute criticality score
        criticality = cost_ratio * (1 + demand_cv) * (1 + lead_time_cv)
        
        return criticality
    
    def calculate_safety_stock(self) -> float:
        """
        Calculate safety stock using combined demand and lead time uncertainty.
        
        Formula: SS = z × √(L × σ²_D + μ² × σ²_L)
        """
        # Calculate variance components
        demand_variance = self.params.lead_time_mean * (self.params.demand_std ** 2)
        lead_time_variance = (self.params.daily_demand ** 2) * (self.params.lead_time_std ** 2)
        
        # Combined standard deviation
        combined_std = math.sqrt(demand_variance + lead_time_variance)
        
        # Safety stock calculation
        safety_stock = self.params.z_score * combined_std
        
        return max(0, safety_stock)
    
    def calculate_eoq(self) -> float:
        """
        Calculate Economic Order Quantity (EOQ).
        
        Formula: Q* = √(2 × D × K / h)
        """
        eoq = math.sqrt((2 * self.params.annual_demand * self.params.ordering_cost) / self.params.holding_cost)
        return eoq
    
    def calculate_reorder_point(self, safety_stock: float) -> float:
        """
        Calculate reorder point with safety stock.
        
        Formula: r = μ × L + SS
        """
        expected_lead_time_demand = self.params.daily_demand * self.params.lead_time_mean
        reorder_point = expected_lead_time_demand + safety_stock
        return reorder_point

print("Priority-Based QR Policy classes defined successfully")

In [ ]:
# Implement the priority-based optimization strategies
class PriorityBasedOptimizer(PriorityBasedQRPolicy):
    """Extended class with optimization strategies and priority queue"""
    
    def __init__(self, params: InventoryParameters):
        super().__init__(params)
        self.strategies = self._initialize_strategies()
    
    def _initialize_strategies(self) -> List[Tuple[float, str, Dict]]:
        """
        Initialize priority queue with different optimization strategies.
        Each strategy has: (priority_score, strategy_type, parameters)
        """
        strategies = []
        
        # Strategy 1: Cost-focused (priority based on criticality score)
        heapq.heappush(strategies, (-self.criticality_score, 'cost_focused', {}))
        
        # Strategy 2: Service-focused (priority based on service level)
        service_priority = -self.params.service_level * 1000
        heapq.heappush(strategies, (service_priority, 'service_focused', {}))
        
        # Strategy 3: Balanced (fixed medium priority)
        heapq.heappush(strategies, (-500, 'balanced', {}))
        
        return strategies
    
    def _cost_focused_optimization(self) -> Tuple[float, float, float]:
        """
        Cost-minimizing strategy: Adjust safety stock based on cost trade-offs.
        """
        q = self.calculate_eoq()
        base_ss = self.calculate_safety_stock()
        
        # Adjust safety stock based on cost ratio
        cost_adjustment = min(1.0, self.params.stockout_cost / (10 * self.params.holding_cost))
        adjusted_ss = base_ss * cost_adjustment
        
        r = self.calculate_reorder_point(adjusted_ss)
        
        return q, r, adjusted_ss
    
    def _service_focused_optimization(self) -> Tuple[float, float, float]:
        """
        Service-level prioritizing strategy: Increase safety stock for high service targets.
        """
        q = self.calculate_eoq()
        ss = self.calculate_safety_stock()
        
        # Add safety buffer for very high service levels
        if self.params.service_level > 0.99:
            ss *= 1.2  # 20% buffer for ultra-high service levels
        
        r = self.calculate_reorder_point(ss)
        
        return q, r, ss
    
    def _balanced_optimization(self) -> Tuple[float, float, float]:
        """
        Balanced strategy: Standard EOQ and safety stock without adjustments.
        """
        q = self.calculate_eoq()
        ss = self.calculate_safety_stock()
        r = self.calculate_reorder_point(ss)
        
        return q, r, ss
    
    def _calculate_total_cost(self, q: float, r: float, safety_stock: float) -> float:
        """
        Estimate total annual cost: holding, ordering, and stockout components.
        """
        # Holding cost: (cycle stock + safety stock) × holding cost per unit
        cycle_stock = q / 2
        holding_cost = (cycle_stock + safety_stock) * self.params.holding_cost
        
        # Ordering cost: (annual demand / order quantity) × ordering cost
        ordering_cost = (self.params.annual_demand / q) * self.params.ordering_cost
        
        # Stockout cost: expected stockouts × stockout cost per unit
        stockout_prob = 1 - self.params.service_level
        expected_stockouts = stockout_prob * self.params.annual_demand / 4  # Approximation
        stockout_cost = expected_stockouts * self.params.stockout_cost
        
        return holding_cost + ordering_cost + stockout_cost
    
    def priority_based_optimization(self, max_iterations: int = 50) -> Dict:
        """
        Main algorithm: Priority queue for strategy selection and optimization.
        """
        best_solution = None
        best_cost = float('inf')
        strategy_results = []
        
        # Process strategies in priority order
        while self.strategies:
            priority, strategy_type, params = heapq.heappop(self.strategies)
            
            # Apply selected strategy
            if strategy_type == 'cost_focused':
                q, r, ss = self._cost_focused_optimization()
            elif strategy_type == 'service_focused':
                q, r, ss = self._service_focused_optimization()
            else:  # balanced
                q, r, ss = self._balanced_optimization()
            
            # Evaluate total cost
            total_cost = self._calculate_total_cost(q, r, ss)
            
            # Store strategy results
            result = {
                'Q': q,
                'r': r,
                'safety_stock': ss,
                'total_cost': total_cost,
                'strategy': strategy_type,
                'priority': -priority  # Convert back to positive for display
            }
            strategy_results.append(result)
            
            # Update best solution if improved
            if total_cost < best_cost:
                best_cost = total_cost
                best_solution = result.copy()
        
        return {
            'best_solution': best_solution,
            'all_strategies': strategy_results,
            'criticality_score': self.criticality_score
        }

print("Priority-based optimization strategies implemented successfully")

In [ ]:
# Initialize CardioStabil problem parameters
cardio_params = InventoryParameters(
    annual_demand=12000,
    lead_time_mean=14,
    lead_time_std=3,
    demand_std=10,
    holding_cost_rate=0.25,
    unit_cost=450,
    ordering_cost=200,
    stockout_cost=2000,
    service_level=0.995
)

# Create optimizer and run priority-based optimization
optimizer = PriorityBasedOptimizer(cardio_params)
results = optimizer.priority_based_optimization()

# Display criticality analysis
print("=== CRITICALITY ANALYSIS ===")
print(f"Criticality Score: {results['criticality_score']:.2f}")

# Calculate criticality components
cost_ratio = cardio_params.stockout_cost / cardio_params.holding_cost
demand_cv = cardio_params.demand_std / cardio_params.daily_demand
lead_time_cv = cardio_params.lead_time_std / cardio_params.lead_time_mean

print(f"Cost Ratio (Stockout/Holding): {cost_ratio:.1f}")
print(f"Demand CV: {demand_cv:.2f}")
print(f"Lead Time CV: {lead_time_cv:.2f}")
print(f"Criticality Formula: {cost_ratio:.1f} × (1 + {demand_cv:.2f}) × (1 + {lead_time_cv:.2f}) = {results['criticality_score']:.2f}")

# Display all strategy results
print("\n=== STRATEGY COMPARISON ===")
strategy_df = pd.DataFrame(results['all_strategies'])
print(strategy_df[['strategy', 'priority', 'Q', 'r', 'safety_stock', 'total_cost']].round(1))

# Display best solution
best = results['best_solution']
print(f"\n=== OPTIMAL SOLUTION ===")
print(f"Strategy: {best['strategy']}")
print(f"Order Quantity (Q): {best['Q']:.0f} units")
print(f"Reorder Point (r): {best['r']:.0f} units")
print(f"Safety Stock: {best['safety_stock']:.0f} units")
print(f"Total Annual Cost: ${best['total_cost']:,.2f}")

In [ ]:
# Create comprehensive visualization of priority-based optimization
def create_priority_visualization(results, params):
    """
    Create a 4-panel visualization showing:
    1. Strategy comparison and costs
    2. Criticality analysis
    3. Priority queue visualization
    4. Parameter sensitivity
    """
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Priority-Based (Q,r) Policy Optimization - CardioStabil', fontsize=16, fontweight='bold')
    
    # Panel 1: Strategy Cost Comparison
    strategies = results['all_strategies']
    strategy_names = [s['strategy'].replace('_', ' ').title() for s in strategies]
    costs = [s['total_cost'] for s in strategies]
    priorities = [s['priority'] for s in strategies]
    
    bars = ax1.bar(strategy_names, costs, color=['#ff6b6b', '#4ecdc4', '#45b7d1'])
    ax1.set_ylabel('Total Annual Cost ($)')
    ax1.set_title('Cost Comparison by Strategy')
    ax1.tick_params(axis='x', rotation=45)
    
    # Add cost labels on bars
    for bar, cost in zip(bars, costs):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                f'${cost:,.0f}', ha='center', va='bottom', fontweight='bold')
    
    # Panel 2: Criticality Analysis
    criticality_components = ['Cost Ratio', 'Demand Variability', 'Lead Time Variability', 'Combined Score']
    criticality_values = [cost_ratio, demand_cv, lead_time_cv, results['criticality_score']]
    
    # Normalize for better visualization
    normalized_values = np.array(criticality_values) / max(criticality_values)
    
    bars2 = ax2.bar(criticality_components, normalized_values, color=['#e74c3c', '#f39c12', '#3498db', '#9b59b6'])
    ax2.set_ylabel('Normalized Value')
    ax2.set_title('Criticality Score Components')
    ax2.tick_params(axis='x', rotation=45)
    
    # Add actual values as text
    for bar, value in zip(bars2, criticality_values):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{value:.2f}', ha='center', va='bottom', fontweight='bold')
    
    # Panel 3: Priority Queue Visualization
    queue_data = []
    for strategy in strategies:
        queue_data.append({
            'Strategy': strategy['strategy'].replace('_', ' ').title(),
            'Priority': strategy['priority'],
            'Q': strategy['Q'],
            'r': strategy['r'],
            'SS': strategy['safety_stock']
        })
    
    queue_df = pd.DataFrame(queue_data)
    
    # Create bubble chart: priority vs parameters
    for i, row in queue_df.iterrows():
        ax3.scatter(row['Priority'], row['Q'], s=row['SS']*2, alpha=0.6, 
                   label=row['Strategy'], edgecolors='black', linewidth=1)
        
    ax3.set_xlabel('Priority Score')
    ax3.set_ylabel('Order Quantity (Q)')
    ax3.set_title('Priority Queue: Strategy Parameters')
    ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax3.grid(True, alpha=0.3)
    
    # Panel 4: Parameter Sensitivity
    # Test how criticality changes with different parameters
    cost_ratios = np.linspace(10, 50, 20)
    criticalities = []
    
    base_cv = (1 + demand_cv) * (1 + lead_time_cv)
    for cr in cost_ratios:
        criticalities.append(cr * base_cv)
    
    ax4.plot(cost_ratios, criticalities, 'purple', linewidth=2)
    ax4.scatter([cost_ratio], [results['criticality_score']], color='red', s=100, zorder=5,
               label=f'Current: {cost_ratio:.1f}')
    ax4.set_xlabel('Cost Ratio (Stockout/Holding)')
    ax4.set_ylabel('Criticality Score')
    ax4.set_title('Criticality vs Cost Ratio')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Create visualization
create_priority_visualization(results, cardio_params)

In [ ]:
# Performance comparison with mathematical formulation baseline
def compare_with_baseline(results, params):
    """
    Compare priority-based heuristic results with mathematical formulation baseline.
    """
    # Mathematical formulation results (from Tier 1)
    baseline = {
        'Q': 207,  # EOQ from mathematical formulation
        'r': 735,  # Reorder point from mathematical formulation
        'safety_stock': 273,  # Safety stock from mathematical formulation
        'strategy': 'Mathematical Formulation'
    }
    
    # Calculate baseline cost
    baseline_cost = optimizer._calculate_total_cost(baseline['Q'], baseline['r'], baseline['safety_stock'])
    baseline['total_cost'] = baseline_cost
    
    # Create comparison dataframe
    comparison_data = results['all_strategies'].copy()
    comparison_data.append(baseline)
    
    comparison_df = pd.DataFrame(comparison_data)
    
    # Calculate performance metrics
    best_heuristic = results['best_solution']
    cost_gap = ((best_heuristic['total_cost'] - baseline_cost) / baseline_cost) * 100
    
    print("=== PERFORMANCE COMPARISON ===")
    print(f"\nMathematical Formulation (Baseline):")
    print(f"   • Q: {baseline['Q']} units")
    print(f"   • r: {baseline['r']} units")
    print(f"   • Safety Stock: {baseline['safety_stock']} units")
    print(f"   • Total Cost: ${baseline_cost:,.2f}")
    
    print(f"\nBest Heuristic Solution ({best_heuristic['strategy'].replace('_', ' ').title()}):")
    print(f"   • Q: {best_heuristic['Q']:.0f} units")
    print(f"   • r: {best_heuristic['r']:.0f} units")
    print(f"   • Safety Stock: {best_heuristic['safety_stock']:.0f} units")
    print(f"   • Total Cost: ${best_heuristic['total_cost']:,.2f}")
    
    print(f"\nPerformance Analysis:")
    print(f"   • Cost Gap: {cost_gap:+.2f}%")
    print(f"   • Q Difference: {best_heuristic['Q'] - baseline['Q']:+.0f} units ({(best_heuristic['Q']/baseline['Q']-1)*100:+.1f}%)")
    print(f"   • r Difference: {best_heuristic['r'] - baseline['r']:+.0f} units ({(best_heuristic['r']/baseline['r']-1)*100:+.1f}%)")
    print(f"   • SS Difference: {best_heuristic['safety_stock'] - baseline['safety_stock']:+.0f} units ({(best_heuristic['safety_stock']/baseline['safety_stock']-1)*100:+.1f}%)")
    
    return comparison_df, cost_gap

# Run comparison
comparison_df, cost_gap = compare_with_baseline(results, cardio_params)

# Display full comparison table
print("\n=== FULL COMPARISON TABLE ===")
display_df = comparison_df[['strategy', 'Q', 'r', 'safety_stock', 'total_cost']].round(1)
display_df['Cost vs Baseline (%)'] = ((display_df['total_cost'] - comparison_df[comparison_df['strategy'] == 'Mathematical Formulation']['total_cost'].iloc[0]) / comparison_df[comparison_df['strategy'] == 'Mathematical Formulation']['total_cost'].iloc[0] * 100).round(1)
print(display_df)

In [ ]:
# Scalability analysis: Test with multiple products
def scalability_analysis():
    """
    Test the priority-based algorithm with multiple product types.
    """
    # Define different product types
    products = {
        'High_Value_Critical': {
            'annual_demand': 12000, 'lead_time_mean': 14, 'lead_time_std': 3,
            'demand_std': 10, 'holding_cost_rate': 0.25, 'unit_cost': 450,
            'ordering_cost': 200, 'stockout_cost': 2000, 'service_level': 0.995
        },
        'Medium_Value_Moderate': {
            'annual_demand': 24000, 'lead_time_mean': 10, 'lead_time_std': 2,
            'demand_std': 15, 'holding_cost_rate': 0.20, 'unit_cost': 120,
            'ordering_cost': 100, 'stockout_cost': 500, 'service_level': 0.95
        },
        'Low_Value_Standard': {
            'annual_demand': 50000, 'lead_time_mean': 7, 'lead_time_std': 1,
            'demand_std': 20, 'holding_cost_rate': 0.15, 'unit_cost': 25,
            'ordering_cost': 50, 'stockout_cost': 100, 'service_level': 0.90
        }
    }
    
    results = []
    
    for product_name, params_dict in products.items():
        # Create parameters
        params = InventoryParameters(**params_dict)
        
        # Run optimization
        optimizer = PriorityBasedOptimizer(params)
        result = optimizer.priority_based_optimization()
        
        # Store results
        best = result['best_solution']
        results.append({
            'Product': product_name.replace('_', ' ').title(),
            'Criticality': result['criticality_score'],
            'Strategy': best['strategy'].replace('_', ' ').title(),
            'Q': best['Q'],
            'r': best['r'],
            'SS': best['safety_stock'],
            'Total_Cost': best['total_cost'],
            'Unit_Cost': params.unit_cost
        })
    
    # Create results dataframe
    results_df = pd.DataFrame(results)
    
    print("=== MULTI-PRODUCT SCALABILITY ANALYSIS ===")
    print(results_df.round(1).to_string(index=False))
    
    # Visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Multi-Product Scalability Analysis', fontsize=14, fontweight='bold')
    
    # Panel 1: Criticality vs Total Cost
    ax1.scatter(results_df['Criticality'], results_df['Total_Cost'], 
               s=results_df['Unit_Cost']/5, alpha=0.7, edgecolors='black')
    for i, row in results_df.iterrows():
        ax1.annotate(row['Product'], (row['Criticality'], row['Total_Cost']), 
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax1.set_xlabel('Criticality Score')
    ax1.set_ylabel('Total Annual Cost ($)')
    ax1.set_title('Criticality vs Total Cost')
    ax1.grid(True, alpha=0.3)
    
    # Panel 2: Strategy Distribution
    strategy_counts = results_df['Strategy'].value_counts()
    ax2.pie(strategy_counts.values, labels=strategy_counts.index, autopct='%1.1f%%',
            colors=['#ff6b6b', '#4ecdc4', '#45b7d1'])
    ax2.set_title('Strategy Selection Distribution')
    
    # Panel 3: Order Quantity vs Unit Cost
    ax3.scatter(results_df['Unit_Cost'], results_df['Q'], alpha=0.7, s=100, edgecolors='black')
    for i, row in results_df.iterrows():
        ax3.annotate(row['Product'][:8], (row['Unit_Cost'], row['Q']), 
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax3.set_xlabel('Unit Cost ($)')
    ax3.set_ylabel('Order Quantity (Q)')
    ax3.set_title('Order Quantity vs Unit Cost')
    ax3.grid(True, alpha=0.3)
    
    # Panel 4: Safety Stock vs Criticality
    ax4.scatter(results_df['Criticality'], results_df['SS'], alpha=0.7, s=100, edgecolors='black')
    for i, row in results_df.iterrows():
        ax4.annotate(row['Product'][:8], (row['Criticality'], row['SS']), 
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax4.set_xlabel('Criticality Score')
    ax4.set_ylabel('Safety Stock (units)')
    ax4.set_title('Safety Stock vs Criticality')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return results_df

# Run scalability analysis
scalability_results = scalability_analysis()

In [ ]:
# Summary and key insights
print("=== PRIORITY-BASED HEURISTIC SUMMARY ===")
print("\n🎯 ALGORITHM CHARACTERISTICS:")
print(f"   • Time Complexity: O(n log n + k) where n=items, k=iterations")
print(f"   • Space Complexity: O(n) for priority queue storage")
print(f"   • Criticality Score: {results['criticality_score']:.2f}")
print(f"   • Optimal Strategy: {best['strategy'].replace('_', ' ').title()}")

print("\n📊 PERFORMANCE METRICS:")
print(f"   • Order Quantity: {best['Q']:.0f} units")
print(f"   • Reorder Point: {best['r']:.0f} units")
print(f"   • Safety Stock: {best['safety_stock']:.0f} units")
print(f"   • Total Cost: ${best['total_cost']:,.2f}")
print(f"   • Cost Gap vs Mathematical: {cost_gap:+.2f}%")

print("\n⚡ ALGORITHM ADVANTAGES:")
print("   • Adaptive strategy selection based on item criticality")
print("   • Efficient priority queue operations (O(log n) per strategy)")
print("   • Multiple optimization strategies in single run")
print("   • Scalable to multi-item inventory systems")
print("   • Real-time decision making capability")

print("\n🔄 STRATEGY COMPARISON:")
for strategy in results['all_strategies']:
    name = strategy['strategy'].replace('_', ' ').title()
    print(f"   • {name}: ${strategy['total_cost']:,.0f} (Priority: {strategy['priority']:.0f})")

print("\n✅ PRIORITY-BASED HEURISTIC COMPLETE")
print("The heuristic provides fast, adaptive solutions with minimal performance gap")
print("compared to mathematical formulation while offering computational efficiency.")